**Create the target table**

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.orders_target (
    order_id INT,
    customer_name STRING,
    order_status STRING,
    amount INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO workspace.default.orders_target VALUES
(101, 'Arun', 'Pending', 500),
(102, 'Priya', 'Completed', 800),
(103, 'John', 'Pending', 300);

num_affected_rows,num_inserted_rows
3,3


**View existing orders**

In [0]:
%sql
SELECT *
FROM workspace.default.orders_target
ORDER BY order_id;

order_id,customer_name,order_status,amount
101,Arun,Pending,500
102,Priya,Completed,800
103,John,Pending,300


**INSERT using MERGE**

**Create a source table**

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.orders_source (
    order_id INT,
    customer_name STRING,
    order_status STRING,
    amount INT
)
USING DELTA;

**Add one new order**

In [0]:
%sql
INSERT INTO workspace.default.orders_source VALUES
(104, 'David', 'Pending', 600);

num_affected_rows,num_inserted_rows
1,1


**View the incoming order**

In [0]:
%sql
SELECT * FROM workspace.default.orders_source;

order_id,customer_name,order_status,amount
104,David,Pending,600


**MERGE with INSERT only**

In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id = source.order_id

WHEN NOT MATCHED THEN
INSERT (
    order_id,
    customer_name,
    order_status,
    amount
)
VALUES (
    source.order_id,
    source.customer_name,
    source.order_status,
    source.amount
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1


**Verify the result**

In [0]:
%sql
SELECT *
FROM workspace.default.orders_target
ORDER BY order_id;

order_id,customer_name,order_status,amount
101,Arun,Pending,500
102,Priya,Completed,800
103,John,Pending,300
104,David,Pending,600


## **2. UPDATE using MERGE**

**Replace the source records**

In [0]:
%sql
TRUNCATE TABLE workspace.default.orders_source;

INSERT INTO workspace.default.orders_source VALUES
(101, 'Arun', 'Completed', 500);

num_affected_rows,num_inserted_rows
1,1


**View the source**

In [0]:
%sql
SELECT * FROM workspace.default.orders_source;

order_id,customer_name,order_status,amount
101,Arun,Completed,500


**MERGE with UPDATE only**

In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
    target.order_status = source.order_status;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql
SELECT *
FROM workspace.default.orders_target
ORDER BY order_id;

order_id,customer_name,order_status,amount
101,Arun,Completed,500
102,Priya,Completed,800
103,John,Pending,300
104,David,Pending,600


**INSERT and UPDATE together**

**Prepare two incoming records**

In [0]:
%sql
TRUNCATE TABLE workspace.default.orders_source;

INSERT INTO workspace.default.orders_source VALUES
(102, 'Priya', 'Refunded', 800),
(105, 'Sarah', 'Pending', 900);

num_affected_rows,num_inserted_rows
2,2


**MERGE with INSERT and UPDATE**

In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
    target.order_status = source.order_status,
    target.amount = source.amount

WHEN NOT MATCHED THEN
INSERT (
    order_id,
    customer_name,
    order_status,
    amount
)
VALUES (
    source.order_id,
    source.customer_name,
    source.order_status,
    source.amount
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


**Verify**

In [0]:
%sql
SELECT *
FROM workspace.default.orders_target
ORDER BY order_id;

order_id,customer_name,order_status,amount
101,Arun,Completed,500
102,Priya,Refunded,800
103,John,Pending,300
104,David,Pending,600
105,Sarah,Pending,900


## DELETE using MERGE

**Prepare the source**

In [0]:
%sql
TRUNCATE TABLE workspace.default.orders_source;

INSERT INTO workspace.default.orders_source VALUES
(103, 'John', 'Delete', 300);

num_affected_rows,num_inserted_rows
1,1


**MERGE with DELETE**

In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id = source.order_id

WHEN MATCHED AND source.order_status = 'Delete'
THEN DELETE;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,1,0


**Verify**

In [0]:
%sql
SELECT *
FROM workspace.default.orders_target
ORDER BY order_id;

order_id,customer_name,order_status,amount
101,Arun,Completed,500
102,Priya,Refunded,800
104,David,Pending,600
105,Sarah,Pending,900


## Common interview question

What happens if multiple source records match the same target record?

**Prepare duplicate source records**

In [0]:
%sql
TRUNCATE TABLE workspace.default.orders_source;

INSERT INTO workspace.default.orders_source VALUES
(101, 'Arun', 'Shipped', 500),
(101, 'Arun', 'Delivered', 500);

num_affected_rows,num_inserted_rows
2,2


**View the duplicate records**

In [0]:
%sql
SELECT *
FROM workspace.default.orders_source;

order_id,customer_name,order_status,amount
101,Arun,Shipped,500
101,Arun,Delivered,500


**Execute MERGE and observe the error**

In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING workspace.default.orders_source AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
    target.order_status = source.order_status;

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-8775103615953246>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'MERGE INTO workspace.default.orders_target AS target\nUSING workspace.default.orders_source AS source\nON target.order_id = source.order_id\n\nWHEN MATCHED THEN\nUPDATE SET\n    target.order_status = source.order_status;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

**Deduplicate**

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW orders_source_dedup AS

SELECT
    order_id,
    customer_name,
    order_status,
    amount
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY
                CASE order_status
                    WHEN 'Delivered' THEN 1
                    WHEN 'Shipped' THEN 2
                    ELSE 3
                END
        ) AS rn
    FROM workspace.default.orders_source
)
WHERE rn = 1;

In [0]:
%sql
SELECT * FROM orders_source_dedup;

order_id,customer_name,order_status,amount
101,Arun,Delivered,500


**Execute MERGE again using deduplicated records**

In [0]:
%sql
MERGE INTO workspace.default.orders_target AS target
USING orders_source_dedup AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
    target.order_status = source.order_status;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql
SELECT *
FROM workspace.default.orders_target
WHERE order_id = 101;

order_id,customer_name,order_status,amount
101,Arun,Delivered,500


In [0]:
%sql

DESCRIBE HISTORY workspace.default.orders_target

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
23,2026-11-07T18:06:01.000Z,76591057851675,thiyaguchandran2021@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4052509571137410),ee479d9a-c869-48e8-9e25-4cece9c30be4,1107-174049-anz9ldc7-v2n,22,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3149, p25FileSize -> 1622, numDeletionVectorsRemoved -> 1, minFileSize -> 1622, numAddedFiles -> 1, maxFileSize -> 1622, p75FileSize -> 1622, p50FileSize -> 1622, numAddedBytes -> 1622)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
22,2026-11-07T18:05:59.000Z,76591057851675,thiyaguchandran2021@gmail.com,MERGE,"Map(predicate -> [""(order_id#14114 = order_id#14095)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(4052509571137410),ee479d9a-c869-48e8-9e25-4cece9c30be4,1107-174049-anz9ldc7-v2n,21,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 1522, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3487, materializeSourceTimeMs -> 607, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 964, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1863)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
21,2026-11-07T17:57:22.000Z,76591057851675,thiyaguchandran2021@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4052509571137410),dc0fc63f-3f77-49c8-960b-909fea449028,1107-174049-anz9ldc7-v2n,20,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1656, p25FileSize -> 1627, numDeletionVectorsRemoved -> 1, minFileSize -> 1627, numAddedFiles -> 1, maxFileSize -> 1627, p75FileSize -> 1627, p50FileSize -> 1627, numAddedBytes -> 1627)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
20,2026-11-07T17:57:20.000Z,76591057851675,thiyaguchandran2021@gmail.com,MERGE,"Map(predicate -> [""(order_id#13106 = order_id#13114)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""(order_status#13116 = Delete)"",""actionType"":""delete""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(4052509571137410),dc0fc63f-3f77-49c8-960b-909fea449028,1107-174049-anz9ldc7-v2n,19,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 1, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1272, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 1, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 758, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 458)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
19,2026-11-07T17:53:50.000Z,76591057851675,thiyaguchandran2021@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4052509571137410),1483f789-ca79-4894-8c9c-72df81f46657,1107-174049-anz9ldc7-v2n,18,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 3189, p25FileSize -> 1